In [1]:
!pip uninstall -y torchaudio
!pip install -q -U "torchao>=0.16.0" transformers peft accelerate datasets rouge-score

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [2]:
import json
from datasets import load_dataset

PROMPT_TEMPLATE = """Extract the innovation problem and solution from the following text as a JSON object with keys 'problem' and 'solution'.
Text: {text}
JSON:"""

def formatting_prompts_func(example):
    text = PROMPT_TEMPLATE.format(text=example['text'])
    label = json.dumps(example['label'], ensure_ascii=False)
    return {"text": text + " " + label, "labels": text + " " + label}

data_files = {
    "train": "train.jsonl",
    "validation": "val.jsonl",
    "test": "test.jsonl"
}

dataset = load_dataset("json", data_files=data_files)
train_dataset = dataset["train"].map(formatting_prompts_func)
val_dataset = dataset["validation"].map(formatting_prompts_func)

print("Dataset successfully loaded and formatted!")

Dataset successfully loaded and formatted!


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load Model directly in fp16/bf16
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto"
)

# Configure LoRA Adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("Qwen 0.5B initialized with LoRA adapters successfully!")

W0721 05:28:06.899000 9926 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0721 05:28:06.935000 9926 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen 0.5B initialized with LoRA adapters successfully!


In [4]:
import json
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model
from rouge_score import rouge_scorer

# ==========================================
# 1. SETUP & MODEL LOADING
# ==========================================
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto"
)
model.config.use_cache = False  # Disable KV cache during fine-tuning

# Configure LoRA Adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("Model and LoRA successfully initialized!")

# ==========================================
# 2. DATASET PREPARATION & TOKENIZATION
# ==========================================
PROMPT_TEMPLATE = """Extract the innovation problem and solution from the following text as a JSON object with keys 'problem' and 'solution'.
Text: {text}
JSON:"""

data_files = {
    "train": "train.jsonl",
    "validation": "val.jsonl",
    "test": "test.jsonl"
}

dataset = load_dataset("json", data_files=data_files)

def tokenize_func(examples):
    prompts = [PROMPT_TEMPLATE.format(text=t) for t in examples['text']]
    targets = [json.dumps(l, ensure_ascii=False) for l in examples['label']]
    full_texts = [p + " " + t for p, t in zip(prompts, targets)]

    # Tokenize full prompt + target string
    model_inputs = tokenizer(full_texts, max_length=512, truncation=True)
    labels = [list(ids) for ids in model_inputs["input_ids"]]

    # Mask prompt tokens with -100 so loss is calculated only on the JSON completion
    for i in range(len(prompts)):
        prompt_ids = tokenizer(prompts[i], max_length=512, truncation=True)["input_ids"]
        prompt_len = len(prompt_ids)
        labels[i][:prompt_len] = [-100] * min(prompt_len, len(labels[i]))

    model_inputs["labels"] = labels
    return model_inputs

print("Tokenizing datasets...")
tokenized_train = dataset["train"].map(tokenize_func, batched=True, remove_columns=dataset["train"].column_names)
tokenized_val = dataset["validation"].map(tokenize_func, batched=True, remove_columns=dataset["validation"].column_names)

# DataCollatorForSeq2Seq handles dynamic padding for both input_ids and labels
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# ==========================================
# 3. TRAINING
# ==========================================
training_args = TrainingArguments(
    output_dir="lora-adapter",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to=[],
    disable_tqdm=False
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    args=training_args,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

# Save final LoRA adapter
model.save_pretrained("lora-adapter")
tokenizer.save_pretrained("lora-adapter")
print("LoRA adapter saved to 'lora-adapter/' successfully!")

# ==========================================
# 4. EVALUATION
# ==========================================
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
model.eval()

def generate_extraction(text):
    prompt = PROMPT_TEMPLATE.format(text=text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("JSON:")[-1].strip()

results = []
print("Evaluating on test set...")
for row in dataset["test"]:
    pred_str = generate_extraction(row['text'])
    label_str = json.dumps(row['label'], ensure_ascii=False)
    score = scorer.score(label_str, pred_str)['rougeL'].fmeasure
    results.append(score)

mean_rouge = np.mean(results)
print("\n=========================================")
print(f"Average ROUGE-L Score on Test Set: {mean_rouge:.4f}")
print("=========================================")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model and LoRA successfully initialized!
Tokenizing datasets...


Map:   0%|          | 0/414 [00:00<?, ? examples/s]

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

Starting training...


Epoch,Training Loss,Validation Loss
1,0.000876,0.001940
2,0.000135,0.001702
3,0.000145,0.002107
4,0.000119,0.001766
5,0.000089,0.001711
6,0.000090,0.001536


LoRA adapter saved to 'lora-adapter/' successfully!
Evaluating on test set...

Average ROUGE-L Score on Test Set: 1.0000


In [5]:
from google.colab import drive
import shutil

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Copy the lora-adapter directory to your Google Drive
shutil.copytree('lora-adapter', '/content/drive/MyDrive/lora-adapter')

print("Adapter successfully copied to your Google Drive!")

Mounted at /content/drive
Adapter successfully copied to your Google Drive!


In [6]:
test_text = """
Our manufacturing plant faces frequent unexpected conveyor belt breakdowns due to motor overheating,
causing hours of downtime every week. To solve this, we installed smart IoT thermal sensors and an automated
cooling system that triggers preemptively when temperatures rise.
"""

print(generate_extraction(test_text))

{"problem": "Our manufacturing plant faces frequent unexpected conveyor belt breakdowns due to motor overheating, causing hours of downtime every week.", "solution": "We installed smart IoT thermal sensors and an automated cooling system that triggers preemptively when temperatures rise."}


In [7]:
# -------------------------------------------------------------
# Test Case 1: Conversational fluff, typos & informal text
# -------------------------------------------------------------
messy_1 = """
so basically yeah, our client onboarding team is losing like 3 hours every day because they gotta manually copy-paste data from customer PDFs into Excel sheets, super tedious and error-prone. to fix this mess we built a tiny python script with OCR that automatically parses the PDF forms and dumps the data right into our database.
"""

# -------------------------------------------------------------
# Test Case 2: Inverted structure (Solution mentioned FIRST, passive voice)
# -------------------------------------------------------------
messy_2 = """
A distributed Redis caching layer with write-through policy was deployed across our edge network. This successfully reduced database query latency by 85% and stopped severe system crashes during flash sales caused by database connection pool exhaustion.
"""

# -------------------------------------------------------------
# Test Case 3: Noisy text with complaints & irrelevant details
# -------------------------------------------------------------
messy_3 = """
Hey team, just wanted to highlight that our support team is completely overwhelmed. Customers are complaining about long hold times (over 20 mins!). Also the office coffee machine was broken yesterday lol. Anyway, to tackle the customer support backlogs, we deployed an AI chatbot powered by RAG to answer tier-1 FAQs instantly, which cut down ticket volume by 40%.
"""

# -------------------------------------------------------------
# Test Case 4: Missing Solution (Negative Test Case)
# -------------------------------------------------------------
messy_4 = """
Our delivery drivers keep missing drop-off windows in crowded downtown areas because standard navigation apps don't account for local double-parking delays and urban construction roadblocks.
"""

# Run all test cases
tests = [
    ("Informal / Fluff", messy_1),
    ("Inverted / Passive Voice", messy_2),
    ("Irrelevant Noise & Complaints", messy_3),
    ("Missing Solution", messy_4)
]

for label, text in tests:
    print(f"=== TEST: {label} ===")
    print(generate_extraction(text))
    print("\n" + "-"*50 + "\n")

=== TEST: Informal / Fluff ===
{"problem": "Our client onboarding team is losing like 3 hours every day because they gotta manually copy-paste data from customer PDFs into Excel sheets, super tedious and error-prone.", "solution": "We built a tiny python script with OCR that automatically parses the PDF forms and dumps the data right into our database."}

--------------------------------------------------

=== TEST: Inverted / Passive Voice ===
{"problem": "A distributed Redis caching layer with write-through policy was deployed across our edge network.", "solution": "This successfully reduced database query latency by 85% and stopped severe system crashes during flash sales caused by database connection pool exhaustion."}

--------------------------------------------------

=== TEST: Irrelevant Noise & Complaints ===
{"problem": "Customers are complaining about long hold times (over 20 mins!)", "solution": "We deployed an AI chatbot powered by RAG to answer tier-1 FAQs instantly, whic

In [8]:
# -------------------------------------------------------------
# Big Test Case 1: Technical Incident Post-Mortem (Multi-paragraph)
# -------------------------------------------------------------
big_1 = """
INCIDENT REPORT #4092 - CLOUD INFRASTRUCTURE
Date: October 14

During our Q3 peak traffic event, the primary PostgreSQL cluster experienced severe write-lock contention, causing query timeouts across our checkout microservices. As API latency spiked above 12,000ms, over 35% of customer transactions failed during a 45-minute window, resulting in approximately $180,000 in lost revenue and heavy social media backlash. The root cause was traced to an unindexed foreign key in the audit logging table combined with synchronized cron jobs performing heavy analytical batch writes.

To permanently resolve this bottleneck, the database reliability team migrated our high-frequency audit logs to an asynchronous Kafka event stream backed by ClickHouse. Furthermore, they decoupled analytical queries from the primary transactional database by establishing read-replicas with automatic connection pooling via PgBouncer.
"""

# -------------------------------------------------------------
# Big Test Case 2: Enterprise Supply Chain & Operations Summary
# -------------------------------------------------------------
big_2 = """
EXECUTIVE SUMMARY: GLOBAL SUPPLY CHAIN PERFORMANCE

Throughout the past fiscal year, our automated distribution centers in Europe encountered severe fulfillment bottlenecks. Due to unexpected cold-chain temperature fluctuations inside transit trucks, temperature-sensitive pharmaceutical shipments suffered a 14% spoilage rate, leading to compliance penalties and significant logistics delays across 12 countries.

In response, our IoT engineering team developed and deployed custom low-power BLE sensor pods equipped with real-time cellular telemetry into all cargo containers. These pods continuously transmit temperature metrics to a cloud platform, automatically triggering auxiliary cooling units whenever thresholds are breached before spoilage can occur.
"""

# -------------------------------------------------------------
# Big Test Case 3: Long Product Development Meeting Transcript
# -------------------------------------------------------------
big_3 = """
TRANSCRIPT EXCERPT - PRODUCT ADOPTION MEETING

Speaker A: "Look, we're getting hammered in the App Store reviews. Users are leaving 1-star reviews claiming the mobile app crashes every time they try to render 3D CAD models on older mobile hardware. Our mobile team tried lowering texture resolutions, but that didn't help much and ruined the graphics quality. The core issue is that local mobile GPUs simply lack the RAM to compute complex mesh transformations on the fly."

Speaker B: "Right, so here is what the graphics team engineered over the last sprint. We transitioned the entire 3D rendering pipeline from local client-side GPU processing to server-side WebGPU streaming. Now, models are rendered on headless cloud instances and streamed directly to the app as H.265 video frames with under 20ms input latency."
"""

# -------------------------------------------------------------
# Big Test Case 4: Dense Academic / R&D Paper Abstract
# -------------------------------------------------------------
big_4 = """
ABSTRACT: AGRICULTURAL YIELD OPTIMIZATION

Smallholder farming communities in semi-arid environments frequently suffer devastating crop loss due to undetected early-stage fungal blight infection, which spreads exponentially before visual symptoms manifest. Traditional laboratory testing of soil and leaves is far too expensive and slow for resource-constrained farmers, often taking weeks to yield results.

To address this diagnostic gap, we engineered a portable, solar-powered spectroscopic biosensor combined with a lightweight computer vision model. The device scans leaf reflectance spectra in the near-infrared band to detect biochemical stress markers 10 days before visual signs appear, enabling targeted localized treatment at a fraction of laboratory costs.
"""

# Run evaluation on all long test cases
big_tests = [
    ("Incident Post-Mortem", big_1),
    ("Supply Chain Summary", big_2),
    ("Meeting Transcript", big_3),
    ("Academic Abstract", big_4)
]

for label, text in big_tests:
    print(f"=== BIG TEST: {label} ===")
    print(generate_extraction(text))
    print("\n" + "="*60 + "\n")

=== BIG TEST: Incident Post-Mortem ===
{"problem": "The primary PostgreSQL cluster experienced severe write-lock contention, causing query timeouts across our checkout microservices.", "solution": "The database reliability team migrated our high-frequency audit logs to an asynchronous Kafka event stream backed by ClickHouse. Furthermore, they decoupled analytical queries from the primary transactional database by establishing read-replicas with automatic connection pooling via PgBouncer."}


=== BIG TEST: Supply Chain Summary ===
{"problem": "Global Supply Chain Performance", "solution": "Our IoT engineering team developed and deployed custom low-power BLE sensor pods equipped with real-time cellular telemetry into all cargo containers.", "value": "These pods continuously transmit temperature metrics to a cloud platform, automatically triggering auxiliary cooling units whenever thresholds are breached before spoilage can occur."}


=== BIG TEST: Meeting Transcript ===
{"problem": "User